In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb


In [2]:
# Create document =>
# from langchain_core.documents import Document

# sample_doc = Document(
#     page_content = "HEllo WoRldD",
#     metadata = {"source":"https://www.google.com"} 
# )

# sample_doc

In [3]:
# # Load Text data =>
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("data/", encoding = "utf-8")
# document = loader.load()
# document

In [4]:
# # Load pdf data =>
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/reserach.pdf")
# document = pdf_loader.load()
# document

# Ingestion Pipeline

In [6]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_core.documents import Document


In [7]:
# Data => Document 

def load_all_pdfs():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            # file location
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs: ",num_docs)
    print("total pages: ",len(all_docs))
    return all_docs
    

In [8]:
all_pdf_documents = load_all_pdfs()

total pdfs:  1
total pages:  21


In [9]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

In [10]:
# Document => chunks 
# !pip install langchain text_splitters

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs


In [12]:

chunks = split_docs(all_pdf_documents)
len(chunks)

244

# Embedding


In [13]:
# Step -2 : chunks =>  Embedding
from sentence_transformers import SentenceTransformer


In [14]:

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading model:", self.model_name)

        self.model = SentenceTransformer(self.model_name)

        print(
            "Embedding dimensions =",
            self.model.get_sentence_embedding_dimension()
        )

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("embeddings shape:", embeddings.shape)
        return embeddings
        

In [15]:
embedding_manager = EmbeddingManager ()

Loading model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions = 384


C:\Users\prath\AppData\Local\Temp\ipykernel_10136\139782092.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.model.get_sentence_embedding_dimension()


# Vector Store 

In [16]:
import chromadb
import uuid


In [17]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "Vector store collection for pdf embedding in RAG"
            }
        )

        print("Initialized the Vector store with collection:", self.collection_name)
        print("Docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents does not match embeddings")

        # Store: ids, embeddings, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            all_metadata.append(metadata)
            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # Add all documents at once
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("Total docs added:", len(documents_content))
        print("Docs in collection:", self.collection.count())

In [18]:

vector_store = VectorStoreManager()

Initialized the Vector store with collection: pdf_documents
Docs in collection: 244


In [19]:
# data => docs => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
Total docs added: 244
Docs in collection: 488


# Retriever Pipeline 

In [20]:
from sklearn.metrics.pairwise import cosine_similarity


In [28]:

class RAGRetriever :
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k = 5, score_threshold = 0.0):
        # data => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search (similar top k value)
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        #cosine similarity 
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            # extracting from doc
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold :
                    retrieved_docs.append({
                        "id" : doc_id,
                        "document" : document,
                        "metadata" : metadata,
                        "similarity_score" : similarity_score,
                        "rank" : i+1
                    })

                print(f"Retrieved {len(retrieved_docs)} dcuments")

        else :
            print("No documents found")

        return retrieved_docs

    
            
        

In [29]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [30]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 1 dcuments
Retrieved 2 dcuments
Retrieved 3 dcuments
Retrieved 4 dcuments
Retrieved 5 dcuments


[{'id': 'doc_48b4702b-dbc6-4aed-9aeb-7e91fe5c1ee5',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'page_label': '1',
   'subject': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'author': '',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'title': '',
   'content_length': 288,
   'trapped': '/False',
   'doc_index': 11,
   'creator': 'LaTeX with hyperref',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'total_pages': 21,
   'source': 'data\\research1.pdf',
   'producer': 'pdfTeX-1.40.25',
   'page': 0,
   'keywords': ''},
  'similarity_score': 0.5370848476886749,
  'rank': 1},
 {'id': 'doc_1c509f73-5809-410d-851b-a6d4ce8a41db',
  'docume

In [32]:
rag_retriever.retrieve("What is encoder decoder?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 0 dcuments
Retrieved 0 dcuments
Retrieved 0 dcuments
Retrieved 0 dcuments
Retrieved 0 dcuments


[]

# Integrate With LLMs

In [ ]:
# OpenAI - GPT

API_KEY_OPENAI = ""

In [33]:
pip install langchain-openai

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 1.3 MB/s eta 0:00:01
   ------------------------ --------------- 1.0/1.7 MB 1.3 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 1.4 MB/s eta 0:00:01
   ------------------------------------ --- 1.6/1.7 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 1.4 MB/s  0:00:01
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/940.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/940.9 kB 1.6 MB/s eta 0:00:01
   --------------------------------- ------ 786.4/940.9 kB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 940.9/940.9 kB 1.5 MB/s  0:00:00

   ---------- --------------------

In [ ]:
from langchain import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-5.4",
    temperature = 0.1,
    max_tokens = 1024
)

In [ ]:
# generate retrieval - output

def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join(doc["document" for doc in results]) if results else ""

    if not context :
        print("No relevant context")

    # Context + query
    prompt = f""" Use given context to generate the answer for the query
                context : {context}
                query : {query} """

    response = llm.invoke(prompt)  #expecting string
    return response.content

    

In [ ]:
answer = generate_output("What is RAG", rag_retriever, llm)

In [ ]:
print(answer)

# Groq 

In [ ]:
# api key
API_KEY_GROQ = ""


In [ ]:
pip install langchain-groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "qwen/qwen3-32b",
    temperature = 0.1,
    max_tokens = 1024
)

In [ ]:
# generate retrieval - output

def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join(doc["document" for doc in results]) if results else ""

    if not context :
        print("No relevant context")

    # Context + query
    prompt = f""" Use given context to generate the answer for the query
                context : {context}
                query : {query} """

    response = llm.invoke([prompt].format(context=context, query=query)  #expecting list
    return response.content

    

In [ ]:
answer = generate_output("What is RAG", rag_retriever, llm)